### MetaData Filtering

* technique used to restrict which documents are searched before or during retrieval based on structured attributes (metadata), rather than searching the entire vector database.

- Metadata is information about a document, not the document's content itself.

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
load_dotenv()

c:\Users\ASUS\.conda\envs\graph-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
loader = PyPDFLoader('attention.pdf')
docs = loader.load()

### Inspecting Metadata

In [13]:
docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszk

In [3]:
print(docs[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [12]:
print(docs[0].page_content[:275])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain


### Chunking

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, #Maximum number of characters per chunk
    chunk_overlap=100 #characters from the previous chunk are repeated in the next chunk to maintain context
)


In [15]:
chunks = splitter.split_documents(docs)

In [16]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16416\614204990.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\ASUS\.conda\envs\graph-env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more de

In [17]:
vectorstore = FAISS.from_documents(chunks, embeddings)

- Metadata

In [18]:
query = "What is attention ?"
results = vectorstore.similarity_search(query, k=2)

In [19]:
for doc in results:
    print(f"Content: {doc.page_content[:50]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 15)

Content: Attention Visualizations
Input-Input Layer5
It
is
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}
---------------
Content: between any two positions in the network. Convolut...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 6, 'page_label': '7'}
--------

- Simple Metadata Filter/(Exact Dict Filter)

In [28]:
page_filter = {"page": 12,'page_label': '13'}

In [29]:
query = "What is attention ?"
results = vectorstore.similarity_search(query, k=2, filter=page_filter)

In [30]:
for doc in results:
    print(f"Content: {doc.page_content[:50]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 15)

Content: Attention Visualizations
Input-Input Layer5
It
is
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}
---------------


- Complex Filter

- $and -> all conditions must be true
- $eq -> exact match (string comparison)
- $gt -> greater than (numeric comparison)

In [31]:
complex_filter = {
    "$and": [
        {"source": {"$eq": "attention.pdf"}},
        {"page": {"$gt": 6}}
    ]
}

In [32]:
query = "What are transformers?"
complex_results = vectorstore.similarity_search(
    query,
    k=2,
    filter=complex_filter
)

In [33]:
for doc in complex_results:
    print(f"Content: {doc.page_content[:50]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 15)

Content: single-precision floating-point capacity of each G...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 7, 'page_label': '8'}
---------------
Content: Table 3: Variations on the Transformer architectur...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 8, 'page_label': '9'}
----------

- Retriver + Pre-Filter

In [34]:
attention_filter = {
    "$and": [
        {"source": {"$eq": "attention.pdf"}},
        {"page": {"$gt": 2}}
    ]
}

In [35]:
filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 1,
        "filter": attention_filter
    }
)

In [36]:
query = "What is the use of vectors?"
retrieved_docs = filtered_retriever.invoke(query)

In [37]:
for doc in retrieved_docs:
    print(f"Content: {doc.page_content[:50]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 15)

Content: the matrix of outputs as:
Attention(Q, K, V ) = so...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 3, 'page_label': '4'}
---------------
